# 데이크스트라 (Dijkstra)

`-` 음이 아닌 가중치를 간선으로 가지는 그래프에서 한 정점과 다른 정점들 사이의 최단 경로를 찾는 알고리즘

## 길 정비하기

- 문제 출처: [정올 8339번](https://jungol.co.kr/problem/8339)

`-` 같은 정점에 위치하더라도 여태까지 오는 도중 몇 개의 길을 고속도로로 바꿨냐에 따라 다르게 취급해야 된다

`-` 따라서 이를 고려해 상태 공간을 설계하자 (정점과 고속도로로 바꾼 길의 개수를 동시에 고려)

`-` 전체 알고리즘의 시간 복잡도는 $O(MK \log MK)$이다

`-` 종종 사용되는 매커니즘이니 잘 알아두자

In [1]:
import heapq
from collections import defaultdict


def dijkstra(graph, source, k):
    inf = float("inf")
    distances = defaultdict(lambda: inf)
    distances[source, 0] = 0
    Q = [(distances[source, 0], source, 0)]
    while Q:
        d_u, u, c = heapq.heappop(Q)
        if distances[u, c] < d_u:
            continue
        for v, w in graph[u]:
            d_v = d_u + w
            if d_v < distances[v, c]:
                distances[v, c] = d_v
                heapq.heappush(Q, (d_v, v, c))
            if c < k and d_u < distances[v, c + 1]:
                distances[v, c + 1] = d_u
                heapq.heappush(Q, (d_u, v, c + 1))
    return distances


def solution():
    N, M, K = map(int, input().split())
    graph = [[] for _ in range(N + 1)]
    for _ in range(M):
        S, E, T = map(int, input().split())
        graph[S].append((E, T))
        graph[E].append((S, T))
    source, sink = 1, N
    distances = dijkstra(graph, source, K)
    answer = distances[sink, K]
    print(answer)


solution()

# input
# 2 1 0
# 1 2 3

 2 1 0
 1 2 3


3


## 로봇

- 문제 출처: [정올 1006번](https://jungol.co.kr/problem/1006)

`-` 명령 $1$을 $1$의 비용으로 $1$칸 움직인 뒤 $0$의 비용으로 $2$칸 더 이동할 수 있는 걸로 치환하자

`-` 상태 공간으로 좌표뿐만 아니라 방향과 $0$의 비용으로 몇 번 더 직진할 수 있는지를 기록하자

`-` 그럼 데이크스트라 알고리즘을 사용해 문제를 해결할 수 있다 (0-1 BFS로도 풀 수 있을 것이다)

`-` 명령 $1$을 $k$마다 분리해서 간선을 설계해도 되는데 귀찮아서 그렇게 안 했다

`-` 근데 오히려 더 어렵게 푼 것 같다

In [2]:
import heapq
from collections import defaultdict


def dijkstra(graph, source):
    inf = float("inf")
    n_rows, n_cols = len(graph), len(graph[0])
    distances = [[[[inf] * 3 for _ in range(4)] for _ in range(n_cols)] for _ in range(n_rows)]
    distances[source[0]][source[1]][source[2]][0] = 0
    drc = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    Q = [(0, *source, 0)]
    while Q:
        d_u, r, c, d, s = heapq.heappop(Q)
        if distances[r][c][d][s] < d_u:
            continue
        left = ((d - 1) + 4) % 4
        right = (d + 1) % 4
        for nd in [left, right]:
            if distances[r][c][nd][0] <= distances[r][c][d][s] + 1:
                continue
            distances[r][c][nd][0] = distances[r][c][d][s] + 1
            heapq.heappush(Q, (distances[r][c][nd][0], r, c, nd, 0))
        dr, dc = drc[d]
        nr, nc = r + dr, c + dc
        is_in_range = 0 <= nr < n_rows and 0 <= nc < n_cols
        if not is_in_range or graph[nr][nc] == 1:
            continue
        if s > 0:
            if distances[r][c][d][s] < distances[nr][nc][d][s - 1]:
                distances[nr][nc][d][s - 1] = distances[r][c][d][s]
                heapq.heappush(Q, (distances[nr][nc][d][s - 1], nr, nc, d, s - 1))
        elif distances[r][c][d][s] + 1 < distances[nr][nc][d][2]:
            distances[nr][nc][d][2] = distances[r][c][d][s] + 1
            heapq.heappush(Q, (distances[nr][nc][d][2], nr, nc, d, 2))
    return distances


def solution():
    M, N = map(int, input().split())
    graph = [list(map(int, input().split())) for _ in range(M)]
    # 북동남서 => 0, 1, 2, 3
    mapping = {1: 1, 2: 3, 3: 2, 4: 0}
    source = list(map(int, input().split()))
    source = source[0] - 1, source[1] - 1, mapping[source[2]]
    sink = list(map(int, input().split()))
    sink = sink[0] - 1, sink[1] - 1, mapping[sink[2]]
    distances = dijkstra(graph, source)
    answer = min([distances[sink[0]][sink[1]][sink[2]][s] for s in range(3)])
    print(answer)


solution()

# input
# 2 2
# 0 0
# 0 0
# 1 1 1
# 2 2 2

 2 2
 0 0
 0 0
 1 1 1
 2 2 2


4
